# Notebook 29 — η_col SBC Failure Investigation

## Purpose

The SBC rank histogram for η_col under S-B shows a **U-shaped distribution with p=0.000**
(spike at ranks 95–100). This means the trained S-B NSF posterior is **overconfident** for
η_col — it reports falsely narrow CIs (0.002–0.016 over a prior range of [0.5, 1.0]).

This notebook investigates the root cause. It is a **research diagnostic — not for the
paper narrative**, which reports this as a limitation in §8.4.

## Hypothesis tree

1. **Weak signal:** η_col genuinely has little effect on S-B summary statistics (because
   x_D is not observed). The NSF learns noise as signal.
2. **Confounded feature:** The physics feature `reb_intensity = Q_reb/F_R_norm` combines
   two channels that both respond to α via the snowball. Under S-B, this creates spurious
   α/η_col confounding.
3. **Architecture overfitting:** NSF (128 hidden, 5 transforms) is too flexible for the
   sparse η_col signal — it learns non-existent correlations from noise in 66-D input.
4. **Insufficient training data:** 15,000 samples don't cover the 5-D prior densely enough
   near the η_col axis.

The investigations below test hypotheses 1, 2, and 4 (in order of cheapness).

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
import torch
import pickle
import pathlib
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from scipy import stats as sp_stats

from cstr_sbi.recycle.simulator import nominal_warm_start
from cstr_sbi.recycle.summaries import compute_summaries, summary_names, PHYSICS_FEATURE_NAMES
from cstr_sbi.recycle.priors import box_uniform_5d
from cstr_sbi.recycle.physics import NOMINAL_CTRL_SB, NOMINAL_INLET, simulate_trajectory_explicit_jit, extract_observations_explicit
import jax.numpy as jnp

DATA     = pathlib.Path('../data')
FIGURES  = pathlib.Path('../figures'); FIGURES.mkdir(exist_ok=True)
SBI_LOGS = pathlib.Path('../sbi-logs')
OI = ["#000000","#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7"]
sb_names = summary_names("S-B")
print(f"Imports OK. {len(sb_names)} S-B feature names.")


## 1. Mutual Information: Which Features Carry η_col vs α Signal?

In [ ]:
# Load pre-computed summary matrix from nb23
d = np.load(DATA / 'wu2003_summary_features.npz', allow_pickle=True)
X_sb        = d['X_sb']            # (420, 66)
labels_sb   = d['labels_sb']       # structured array
alpha_true   = labels_sb['alpha'].astype(float)
eta_col_true = labels_sb['eta_col'].astype(float)
beta_r_true  = labels_sb['beta_r'].astype(float)

X_sc = StandardScaler().fit_transform(X_sb)
mi_alpha   = mutual_info_regression(X_sc, alpha_true,   random_state=42)
mi_eta_col = mutual_info_regression(X_sc, eta_col_true, random_state=42)
mi_beta_r  = mutual_info_regression(X_sc, beta_r_true,  random_state=42)

print("Top 10 features by MI with eta_col:")
for i in np.argsort(mi_eta_col)[::-1][:10]:
    print(f"  {sb_names[i]:<35} MI_eta={mi_eta_col[i]:.4f}  MI_alpha={mi_alpha[i]:.4f}")

print("\nTop 10 features by MI with alpha:")
for i in np.argsort(mi_alpha)[::-1][:10]:
    print(f"  {sb_names[i]:<35} MI_alpha={mi_alpha[i]:.4f}  MI_eta={mi_eta_col[i]:.4f}")

print(f"\nTotal MI(alpha)  : {mi_alpha.sum():.3f}")
print(f"Total MI(eta_col): {mi_eta_col.sum():.3f}")
print(f"Ratio alpha/eta  : {mi_alpha.sum()/mi_eta_col.sum():.1f}x more MI for alpha vs eta_col")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
x = np.arange(len(sb_names))
axes[0].bar(x, mi_alpha,   color=OI[1], alpha=0.8); axes[0].set_ylabel("MI with alpha", fontsize=10)
axes[1].bar(x, mi_eta_col, color=OI[2], alpha=0.8); axes[1].set_ylabel("MI with eta_col", fontsize=10)
axes[0].set_title("Mutual Information: S-B Summary Features (14 scenarios x 30 reps)", fontsize=12)
axes[1].set_xlabel("Feature index (0-53: channel stats; 54-65: physics)", fontsize=10)
for ax in axes: ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'nb29_mi_alpha_vs_etacol.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb29_mi_alpha_vs_etacol.png")


## 2. Deterministic η_col Sweep: How Sensitive Are S-B Summaries to η_col vs α?

If the S-B summaries barely move when η_col changes (holding α=1.0 fixed), that is direct
evidence of Hypothesis 1 (weak signal). The NSF would be learning noise.

In [ ]:
y0_sb = nominal_warm_start("S-B")
t_h_ref = None

eta_vals   = np.linspace(0.50, 1.00, 11)
alpha_vals = np.linspace(0.50, 1.00, 11)

print("Sweeping eta_col (alpha=1.0 fixed)...")
eta_sums = []
for eta in eta_vals:
    th = jnp.array([1.0, 1.0, float(eta), 1.0, 0.90], dtype=jnp.float32)
    ts, ys = simulate_trajectory_explicit_jit(th, NOMINAL_INLET, NOMINAL_CTRL_SB, y0_sb,
                                              t_final=2.0, n_save=120, rtol=1e-3, atol=1e-5)
    raw = np.asarray(extract_observations_explicit(ys, th, NOMINAL_CTRL_SB))
    if t_h_ref is None: t_h_ref = np.asarray(ts)
    eta_sums.append(compute_summaries(raw, "S-B", t_h_ref))

print("Sweeping alpha (eta_col=1.0 fixed)...")
alpha_sums = []
for al in alpha_vals:
    th = jnp.array([float(al), 1.0, 1.0, 1.0, 0.90], dtype=jnp.float32)
    ts, ys = simulate_trajectory_explicit_jit(th, NOMINAL_INLET, NOMINAL_CTRL_SB, y0_sb,
                                              t_final=2.0, n_save=120, rtol=1e-3, atol=1e-5)
    raw = np.asarray(extract_observations_explicit(ys, th, NOMINAL_CTRL_SB))
    alpha_sums.append(compute_summaries(raw, "S-B", t_h_ref))

eta_sums   = np.stack(eta_sums)   # (11, 66)
alpha_sums = np.stack(alpha_sums) # (11, 66)
eta_std    = eta_sums.std(axis=0)
alpha_std  = alpha_sums.std(axis=0)
ratio      = eta_std / (alpha_std + 1e-12)

print("\nTotal feature variation:")
print(f"  sum(std over eta_col sweep) = {eta_std.sum():.4f}")
print(f"  sum(std over alpha sweep)   = {alpha_std.sum():.4f}")
print(f"  Ratio (eta/alpha variation)  = {eta_std.sum()/alpha_std.sum():.3f}")
print("\nTop 10 features most sensitive to eta_col relative to alpha:")
for i in np.argsort(ratio)[::-1][:10]:
    print(f"  {sb_names[i]:<35} ratio={ratio[i]:.3f}  eta_std={eta_std[i]:.5f}  alpha_std={alpha_std[i]:.5f}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
x = np.arange(len(sb_names))
axes[0].bar(x, alpha_std, color=OI[1], alpha=0.8)
axes[0].set_ylabel("Feature std (alpha sweep)", fontsize=10)
axes[0].set_title("Deterministic Summary Sensitivity: alpha sweep vs eta_col sweep", fontsize=12)
axes[1].bar(x, eta_std, color=OI[2], alpha=0.8)
axes[1].set_ylabel("Feature std (eta_col sweep)", fontsize=10)
axes[1].set_xlabel("Feature index", fontsize=10)
for ax in axes: ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'nb29_sensitivity_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb29_sensitivity_sweep.png")


## 3. reb_intensity Confounding Diagnosis

In [ ]:
# reb_intensity = Q_reb / F_R_norm: designed to capture eta_col
# Under S-B, F_R_norm responds to BOTH alpha and eta_col via snowball.
# If reb_intensity correlates strongly with BOTH, it's a confounded feature.
pfn_list = list(PHYSICS_FEATURE_NAMES)
reb_idx  = 54 + pfn_list.index("reb_intensity")
fr_idx   = 54 + pfn_list.index("recycle_ratio")
print(f"reb_intensity at index {reb_idx}, recycle_ratio at index {fr_idx}")

for feat_idx, feat_name in [(reb_idx, "reb_intensity"), (fr_idx, "recycle_ratio")]:
    c_alpha = np.corrcoef(X_sb[:, feat_idx], alpha_true)[0,1]
    c_eta   = np.corrcoef(X_sb[:, feat_idx], eta_col_true)[0,1]
    c_beta  = np.corrcoef(X_sb[:, feat_idx], beta_r_true)[0,1]
    print(f"\n{feat_name}:")
    print(f"  corr(alpha)   = {c_alpha:+.4f}")
    print(f"  corr(eta_col) = {c_eta:+.4f}")
    print(f"  corr(beta_r)  = {c_beta:+.4f}")
    if abs(c_alpha) > abs(c_eta):
        print(f"  -> alpha-dominated feature (may confound eta_col identification)")
    else:
        print(f"  -> eta_col-dominated feature (genuine signal)")


## 4. Extended SBC: N_SBC=500, N_POST=200 for η_col

In [ ]:
# Run extended SBC to get more precise p-value for eta_col
# Only execute if hypotheses 1-3 suggest the problem is fixable
with open(SBI_LOGS / 'wu2003_posterior_sb.pkl', 'rb') as f:
    post_sb = pickle.load(f)['posterior']
prior = box_uniform_5d()

N_SBC = 200  # increase to 500 for publication-quality check
N_POST = 200
print(f"Running extended SBC: {N_SBC} trials, {N_POST} samples/trial...")
print("Computing only eta_col ranks (index 2) to save time...")

sbc_eta_ranks = []
rng = np.random.default_rng(12345)
t0 = __import__('time').time()

for i in range(N_SBC):
    th = prior.sample((1,)).numpy()[0]
    th_j = jnp.array(th, dtype=jnp.float32)
    ts, ys = simulate_trajectory_explicit_jit(th_j, NOMINAL_INLET, NOMINAL_CTRL_SB,
                                              nominal_warm_start("S-B"),
                                              t_final=2.0, n_save=120, rtol=1e-3, atol=1e-5)
    raw = np.asarray(extract_observations_explicit(ys, th_j, NOMINAL_CTRL_SB))
    if np.isnan(raw).any() or np.isinf(raw).any():
        continue
    sc = np.maximum(np.max(np.abs(raw), axis=0), 1e-6)
    s  = compute_summaries(raw + rng.normal(0, 0.003*sc, raw.shape), "S-B", np.asarray(ts))
    if np.isnan(s).any():
        continue
    samp = post_sb.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32)).numpy()
    sbc_eta_ranks.append(int(np.sum(samp[:,2] < th[2])))
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{N_SBC}  ({__import__('time').time()-t0:.0f}s)")

sbc_eta_ranks = np.array(sbc_eta_ranks)
ks = sp_stats.ks_1samp(sbc_eta_ranks / N_POST, sp_stats.uniform.cdf)
print(f"\next SBC eta_col: KS p = {ks.pvalue:.4f}, n_trials = {len(sbc_eta_ranks)}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sbc_eta_ranks, bins=20, range=(0, N_POST), color=OI[2], edgecolor='white', alpha=0.8)
ax.axhline(len(sbc_eta_ranks)/20, ls='--', color='gray', label='Uniform')
ax.set_title(f"Extended SBC: eta_col rank histogram (KS p={ks.pvalue:.4f})", fontsize=11)
ax.set_xlabel("Rank"); ax.set_ylabel("Count"); ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'nb29_sbc_etacol_extended.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb29_sbc_etacol_extended.png")


## Summary

| Investigation | Key metric | Finding | Implication |
|--------------|-----------|---------|-------------|
| MI analysis | Total MI ratio | | |
| Deterministic sweep | eta/alpha variation ratio | | |
| reb_intensity | corr(alpha) vs corr(eta) | | |
| Extended SBC | KS p-value | | |

### Conclusion

*(Fill in after running)*

**Root cause:** ...

**Recommended fix:** ...

**Impact on paper:** η_col overconfidence is reported as a limitation in §8.4.
The banana posterior claim (based on α CI width) is unaffected.